# 03주 · 관 마찰손실·국부손실

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 마찰손실 / 국부손실

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week03/CE272_W03.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week03) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 주제별 실습 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: Colebrook 반복·급확대 손실


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

<a id="s01"></a>

## Moody·Colebrook과 관 마찰손실

**먼저 예측**: 조도 0.12 mm를 0.12 m로 넣으면 어느 무차원 입력이 달라지는가?

**문제**: 직경 0.2 m, 조도 0.12 mm인 관에 물이 0.05 m³/s로 흐른다. 길이 1,000 m의 마찰손실수두를 구하시오. 동점성계수는 원본과 같이 1.004×10⁻⁶ m²/s를 사용하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
Q, D, L, epsilon, nu = .05, .2, 1000., .12/1000, 1.004e-6
V = Q/hyd.area_pipe(D)
Re = hyd.reynolds(Q,D,nu)
f = hyd.darcy_factor(Re,epsilon/D)
colebrook_residual = 1/np.sqrt(f)+2*np.log10(epsilon/(3.7*D)+2.51/(Re*np.sqrt(f)))
pd.Series({'V_m_s':V,'Re':Re,'relative_roughness':epsilon/D,'Darcy_f':f,
           'headloss_m':hyd.pipe_loss(Q,L,D,nu,epsilon),'Colebrook_residual':colebrook_residual})

**검토 문제**: 매끈한 관의 결과와 비교하시오. 마찰계수를 일정하게 놓은 유량 민감도와, 매번 Reynolds 수에 따라 다시 구한 민감도의 차이를 설명하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분


<a id="s02"></a>

## 급확대 손실과 기준속도

**먼저 예측**: 큰 관 단면의 속도만으로 급확대 손실을 계산할 수 있는가?

**문제**: 직경 0.1 m인 관이 0.5 m로 급확대된다. 유량이 0.2 m³/s일 때 급확대 손실수두를 구하시오. 같은 작은 관이 큰 수조로 유입되는 경우와 비교하시오. 마찰손실은 무시하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
Q, D1, D2 = .2, .1, .5
V1, V2 = Q/hyd.area_pipe(D1), Q/hyd.area_pipe(D2)
h_expansion = (V1-V2)**2/(2*hyd.G)
K_based_on_V1 = (1-(D1/D2)**2)**2
h_K = K_based_on_V1*V1**2/(2*hyd.G)
pd.Series({'V1_m_s':V1,'V2_m_s':V2,'K_V1':K_based_on_V1,
           'expansion_loss_m':h_expansion,'K_formula_m':h_K,
           'large_reservoir_exit_loss_m':V1**2/(2*hyd.G)})

**검토 문제**: K의 기준속도를 바꾸면 K도 달라져야 하는 이유를 설명하시오. 국부손실이라는 명칭이 작은 손실을 뜻하는지 검토하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분


## 학습 기록

Darcy 계수, 조도 단위, 기준속도와 에너지 단면을 명시하여 손실을 설명하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록


In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')